# Halim Toddler — Colab (fresh v4 flow)

**Before you start:** Runtime → **Change runtime type → T4 GPU**

## Mac (once per train batch)
```bash
cd ~/Downloads/tradingbot
./scripts/halim_colab_ready.sh    # full rebuild — safe, does not delete gold
```
Upload **`halim_sft.zip`** from repo root in Cell 2 (not to Drive).

## Where things live

| Asset | Location |
|-------|----------|
| `halim_sft.zip` | Upload to **Colab** (Cell 2) |
| Training | `/content/toddler_v1` (fast disk) |
| Finished `halim_toddler_vN.zip` | **Drive** `My Drive/Halim/` (Cell 4) |
| Mac install | `./scripts/halim_apply_colab_checkpoint.sh` |

Run cells **1 → 2 → 3 → 4** in order. Cell 3 = **batch 8** fresh train (~6h on T4).

In [ ]:
# Cell 1 — Mount Drive (version storage only)
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/Halim'
os.makedirs(WORK, exist_ok=True)
os.environ['HALIM_WORK'] = WORK
print('Drive folder:', WORK)
!ls -la "$WORK"

In [ ]:
# Cell 2 — Upload halim_sft.zip to Colab + setup
!pip install -q transformers peft trl datasets accelerate bitsandbytes

from google.colab import files
import os, shutil, zipfile
from pathlib import Path

# Clean slate — no leftover scripts or partial train from old session
for stale in ('sft', 'toddler_v1', 'train_toddler_colab.py', 'colab_drive_setup.py'):
    p = Path('/content') / stale
    if p.is_dir():
        shutil.rmtree(p)
    elif p.is_file():
        p.unlink()

print('Pick halim_sft.zip from Mac (~/Downloads/tradingbot/halim_sft.zip)')
uploaded = files.upload()
sft_zip = None
for name, data in uploaded.items():
    dest = Path('/content') / name
    dest.write_bytes(data)
    if name.endswith('.zip'):
        sft_zip = dest
    print('Colab file:', dest)

if sft_zip is None:
    sft_zip = next(Path('/content').glob('halim_sft*.zip'), None)
if sft_zip is None:
    raise FileNotFoundError('Upload halim_sft.zip via the widget above')

with zipfile.ZipFile(sft_zip, 'r') as zf:
    zf.extractall('/content')

WORK = Path(os.environ['HALIM_WORK'])
if not WORK.is_dir():
    raise FileNotFoundError('Run Cell 1 first — Drive needed for output zip naming')

%cd /content
!python colab_drive_setup.py

In [ ]:
# Cell 3 — Fresh train (T4 sweet spot: batch 8, ~6h)
%cd /content
import os, shutil
from pathlib import Path

adapter = Path('toddler_v1/lora_adapter')
if adapter.exists():
    shutil.rmtree(adapter)

os.environ['HALIM_OUT_DIR'] = '/content/toddler_v1'
os.environ['HALIM_FRESH_TRAIN'] = 'true'
os.environ['HALIM_CONTINUE_LORA'] = 'false'
os.environ['HALIM_FAST_PATH'] = 'auto'
os.environ['HALIM_MAX_POWER'] = 'false'
os.environ['HALIM_BATCH_SIZE'] = '8'
os.environ['HALIM_GRAD_ACCUM'] = '1'
os.environ['HALIM_FP16'] = 'false'
os.environ['HALIM_BF16'] = 'false'

!python train_toddler_colab.py

In [ ]:
# Cell 4 — Save halim_toddler_vN.zip to Drive + Mac install
import json, shutil, subprocess
from pathlib import Path

WORK = Path('/content/drive/MyDrive/Halim')
state_path = WORK / 'halim_colab_state.json'
state = json.loads(state_path.read_text()) if state_path.is_file() else {}
out_name = state.get('next_output_zip', 'halim_toddler_v4.zip')

src = Path('/content/toddler_v1')
merged = src / 'merged'
if not merged.is_dir():
    raise FileNotFoundError('Training not finished — merged/ missing. Wait for Cell 3.')

dst = WORK / 'toddler_v1'
shutil.copytree(src, dst, dirs_exist_ok=True)

subprocess.run(['zip', '-r', out_name, 'toddler_v1'], cwd=str(WORK), check=True)
print('Saved:', WORK / out_name)
print()
print('=== Mac ===')
print('1. Download', out_name, 'to ~/Downloads (or sync Drive Halim/)')
print('2. ./scripts/halim_apply_colab_checkpoint.sh')
print('   (or restart HANOON — HALIM_AUTO_INSTALL_COLAB=true)')
print('3. After install: ./scripts/halim_record_train.sh')